# 01 — Coleta Atlas (GET) — documentação da disciplina

**Projeto:** Preditor de Falhas ML (Grupo 16)
**Disciplinas:** ED2 + Redes + APS

> Este notebook é a **documentação / demonstração** pedida pela professora.
> **Não** é o pipeline de produção.
> O **core** do collector fica no pacote Python: `fetch_measurement_results`
> (`src/preditor_de_falhas_ml/atlas.py`). A CLI `getResults` e a Lambda (S1.7)
> reutilizam a mesma função — não este notebook.
> **Não** criamos medição aqui (sem POST). Só lemos `/results/` de um `msm_id` já existente.


## Equipe

| Integrante | Papel nesta entrega |
|---|---|
| Guilherme Leite Tavares | [preencher] |
| Alexandre Tiago de Oliveira | [preencher] |
| Ingrid Ferreira de Sousa | [preencher] |
| Kauan Garcia Dias de Oliveira | [preencher] |
| Lucas Eduardo Malachias Bagatela | [preencher] |
| Stephanie Vitoria Bessa dos Santos | [preencher] |


## Decisões (justificativa)

| Decisão | Escolha | Por quê |
|---|---|---|
| Verbo HTTP | **GET** `/measurements/{msm_id}/results/` | Collector só lê results; **POST** (criar medição) é S1.6 |
| Onde está o código | Pacote `src/preditor_de_falhas_ml/atlas.py` | Core compartilhado: CLI + Lambda S1.7 importam a mesma função |
| Papel deste notebook | Documentação da disciplina + demo pontual | Exigência da professora; **não** é o pipeline AWS |
| `msm_id` | Medição **já existente** | Sem POST / sem gastar crédito de criação nesta demo |
| Janela | `start` / `stop` Unix UTC (curta) | Smoke controlado; mesmos params da CLI `getResults` |
| Tipo da medição | ping (lido, não criado) | Dataset do preditor; traceroute entra em S1.6 |
| Formato na demo | DataFrame pandas **bruto** | Sem agregação, sem `status_real`, sem treino ML |
| Persistência local | opcional JSONL + metadata em `data/raw/` | Produção grava no S3 na S1.7 |


## Params

Os mesmos da função do pacote e da CLI `getResults`:

- `msm_id` — id de uma medição Atlas já existente
- `start` / `stop` — janela Unix UTC (inclusiva no contrato da API)

Na raiz do repo: `uv sync`.
Defina `RIPE_ATLAS_API_KEY` no ambiente (ou `.env` — **não** commitado).
Opcional: `DEMO_MSM_ID` com um id real, ou edite `msm_id` na célula abaixo.


In [ ]:
import os
from datetime import datetime, timezone

from preditor_de_falhas_ml import fetch_measurement_results

# --- params da demo (ajuste) ---
msm_id = int(os.environ.get("DEMO_MSM_ID", "0"))  # troque por um msm_id real
stop = int(datetime.now(tz=timezone.utc).timestamp())
start = stop - 2 * 3600  # janela curta (2 h) para smoke
GRAVAR_RAW = False  # True só se quiser evidência local em data/raw/

api_key = os.environ.get("RIPE_ATLAS_API_KEY", "")

{"msm_id": msm_id, "start": start, "stop": stop, "GRAVAR_RAW": GRAVAR_RAW}


## Chamada ao core (GET)

A célula abaixo **importa** `fetch_measurement_results` do pacote.
Não usa `requests` nas células e **não** chama `get_data` (POST).


In [ ]:
if not api_key:
    raise RuntimeError("Defina RIPE_ATLAS_API_KEY no ambiente antes de rodar a demo.")
if msm_id <= 0:
    raise RuntimeError("Defina DEMO_MSM_ID ou edite msm_id com um id já existente.")

df_raw = fetch_measurement_results(
    api_key,
    msm_id,
    start=start,
    stop=stop,
)

df_raw.head()


In [ ]:
df_raw.info()
len(df_raw)


## Persistência local (opcional)

`append_data` só entra se `GRAVAR_RAW` for `True`.
Só para evidência da disciplina. Produção grava no S3 (S1.7).


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

meta = {
    "msm_id": msm_id,
    "start": start,
    "stop": stop,
    "utc_run": datetime.now(tz=timezone.utc).isoformat(),
    "n_rows": int(len(df_raw)),
    "columns": list(map(str, df_raw.columns)),
    "note": "demo notebook; core = fetch_measurement_results",
}

if GRAVAR_RAW:
    from preditor_de_falhas_ml import append_data

    output_dir = Path("data/raw")
    output_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    meta_path = output_dir / f"ripe_atlas_m{msm_id}_{stamp}_metadata.json"
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    out = append_data(df_raw, output_dir=output_dir)
    {
        "jsonl": str(out),
        "metadata": str(meta_path),
        "bytes_jsonl": out.stat().st_size if out.exists() else 0,
    }
else:
    meta


## Checklist

- [ ] Equipe preenchida (papéis)
- [ ] Decisões justificadas (GET vs POST; core = pacote; notebook = doc)
- [ ] GET via `from preditor_de_falhas_ml import fetch_measurement_results`
- [ ] Sem POST / sem `requests` / sem `get_data` nas células
- [ ] Params: `msm_id`, `start`, `stop`
- [ ] DataFrame `head()` / `info()` sem agregação nem rótulo
- [ ] (Opcional) raw + metadata em `data/raw/` via `append_data`
- [ ] Claro para a banca: **notebook = doc**; **produção = pacote + Lambda**
